# US Tornado Data: Getting Started

Load and inspect **2010–2025, release v2.0.0 / Kaggle dataset version 5** using seven consolidated Parquet tables (about **24.1 MB**, including the annual summary).

The collection combines SPC tornado tracks, NCEI Storm Events, NOAA Event Footprint Catalog damage regions, Census county population/housing estimates, and a simplified 2020 Census county map. Footprints replace standalone DAT survey tables in v2; earlier dataset versions retain those surveys.

This notebook demonstrates loading, coverage, missing values, and maps. The sources have different row meanings and have **not** been matched into one tornado-level modeling table.

[Dataset](https://www.kaggle.com/datasets/jakevanslyke/us-tornado-data-2010-2025/versions/5) · [Field dictionary](https://github.com/jakeryderv/us-tornado-data-2010-2025/blob/a84ca6f4f8382788b7952e4b63d151696684ea59/docs/ANALYSIS.md) · [Release receipt](https://github.com/jakeryderv/us-tornado-data-2010-2025/blob/4ac992e716b278f4e99c3555a8c54fb288f3c60c/release/v2.0.0.json)

## Load the pinned release with kagglehub

On Kaggle, attach **dataset version 5 before saving/running a version**. `kagglehub` accesses Kaggle's shared dataset cache; outside Kaggle, it downloads individual requested files into a local cache. The same code runs in either environment, without hard-coded `/kaggle/input` paths or repository helper modules.

For local use, install `kagglehub`, `pandas`, `pyarrow`, `geopandas`, `matplotlib`, and `ipykernel`. Kaggle supplies these packages. GeoPandas loads the footprint and boundary GeoParquet files with their CRS intact.

The release manifest is checked against the published SHA-256, and each file used below is checked against that manifest. This verifies our **analysis subset**, not every file in the full collection. We do not download or extract `release.zip.bin`.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform

import kagglehub
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display

get_ipython().run_line_magic("matplotlib", "inline")

DATASET = "jakevanslyke/us-tornado-data-2010-2025/versions/5"
RELEASE = "v2.0.0"
MANIFEST_SHA256 = "d6cf2577f31663c568878e9047e70ec0e26aa4c38f41245aa5620dfa423001d5"

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

manifest_path = Path(kagglehub.dataset_download(DATASET, path="release_manifest.json"))
if sha256(manifest_path) != MANIFEST_SHA256:
    raise ValueError("Release manifest mismatch: attach Kaggle dataset version 5.")
manifest = json.loads(manifest_path.read_text())
assert manifest["version"] == RELEASE
release_files = {entry["path"]: entry for entry in manifest["files"]}
verified_paths = {}

def verified_file(name):
    if name not in verified_paths:
        path = Path(kagglehub.dataset_download(DATASET, path=name))
        if sha256(path) != release_files[name]["sha256"]:
            raise ValueError(f"Checksum mismatch: {name}")
        verified_paths[name] = path
    return verified_paths[name]

print(f"Release: {RELEASE}; Kaggle dataset version: 5")
print(f"Python {platform.python_version()}, pandas {pd.__version__}, GeoPandas {gpd.__version__}")

In [ ]:
analysis_manifest = json.loads(verified_file("analysis/manifest.json").read_text())
geometry_tables = {"tornado_footprints.parquet", "county_boundaries.parquet"}
tables = {}
for entry in analysis_manifest["files"]:
    name = entry["path"]
    path = verified_file(f"analysis/{name}")
    if name.endswith(".parquet"):
        reader = gpd.read_parquet if name in geometry_tables else pd.read_parquet
        tables[Path(name).stem] = reader(path)
    else:
        tables[Path(name).stem] = pd.read_csv(path)
    assert len(tables[Path(name).stem]) == entry["rows"], name

spc = tables["tornadoes"]
events = tables["storm_events"]
fatalities = tables["storm_fatalities"]
locations = tables["storm_locations"]
footprints = tables["tornado_footprints"]
counties = tables["county_context"]
boundaries = tables["county_boundaries"]
annual = tables["annual_summary"]

units = {
    "tornadoes": "SPC tornado track",
    "storm_events": "NCEI tornado event/county segment",
    "storm_fatalities": "NCEI tornado-related fatality record",
    "storm_locations": "NCEI event location record",
    "tornado_footprints": "EFC damage region (possibly nested)",
    "county_context": "County-year population/housing estimates",
    "county_boundaries": "Generalized 2020 county boundary",
    "annual_summary": "Source record totals by year",
}
inventory = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns), "row_meaning": units[name]}
    for name, frame in tables.items()
])
display(inventory)
print(f"Verified table bytes: {sum(entry['bytes'] for entry in analysis_manifest['files']) / 1e6:.1f} MB")

## SPC: tracks, EF ratings, and missing values

SPC is the primary tornado catalog. Unknown EF ratings remain null in `ef_rating`; original `spc_rating_code=-9` is preserved. Unknown does not mean EF0. Track endpoints approximate a path and are not detailed damage geometry.

These plots describe recorded events and ratings. Changes over time also reflect reporting and assessment practices. The 2010–2025 period is within the EF era; 2010 was not the scale transition date.

In [ ]:
assert len(spc) == 20164 and spc["tornado_id"].is_unique
assert spc["ef_rating"].isna().sum() == 1525
ratings = spc["ef_rating"].value_counts().reindex(range(6), fill_value=0)
ratings.index = [f"EF{i}" for i in ratings.index]
ratings.loc["Unknown"] = spc["ef_rating"].isna().sum()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
spc.groupby("year").size().plot.bar(ax=axes[0], color="#277e8e")
axes[0].set(title="SPC tracks by year", xlabel="Year", ylabel="Track records")
ratings.plot.bar(ax=axes[1], color=["#277e8e"] * 6 + ["#999999"])
axes[1].set(title="Recorded EF rating", xlabel="Rating", ylabel="Track records")
fig.tight_layout()
plt.show()
display(spc[["tornado_id", "year", "state", "ef_rating", "path_length_miles", "path_width_yards"]].head())
display(spc.isna().sum().loc[lambda values: values > 0].rename("missing_rows").to_frame())

## NCEI: event narratives and related tables

Details are event/county segments, with separate fatality and location records. `event_id` links these **NCEI tables to each other**; it does not establish matches to SPC or EFC identifiers. Keep one-to-many relationships explicit so joins do not multiply event counts unexpectedly.

NCEI parsed dates remain local, accompanied by `source_timezone`. Do not treat them as UTC. Narratives and damage fields can reveal the target EF rating.

In [ ]:
assert fatalities["event_id"].isin(events["event_id"]).all()
assert locations["event_id"].isin(events["event_id"]).all()
display(events[["event_id", "year", "state_name", "county_zone_name", "ef_rating",
                "begin_datetime_local", "source_timezone", "event_narrative"]].head())
display(fatalities.head(3))
display(locations.head(3))
display(annual[["year", "spc_tracks", "ncei_tornado_details_rows",
                "ncei_tornado_fatalities_rows", "ncei_tornado_locations_rows"]])

## Footprint Catalog: geometry and coverage

EFC prioritizes DAT survey geometry and supplements it with Storm Events (`source="SED"`) reconstructions. Its rows are **damage regions, not unique tornadoes**: a tornado may have nested or overlapping regions. Coverage counts below are not a fraction of SPC tornadoes matched.

`footprint_id` uniquely identifies a catalog region. `parents`/`children` describe source relationships; SED event IDs are generated catalog identifiers. No cross-source join is supplied.

Raw ratings, dates and width sentinels remain intact. `ef_rating` and `max_ef_rating` are separate nullable helpers; neither automatically replaces the SPC label. `path_width_yards` is null for unusable widths, including placeholder `0.99` and zero. `*_datetime_utc` helpers parse EFC dates; `source_year` follows annual file partitions, which can cross UTC New Year.

In [ ]:
assert len(footprints) == 24858 and footprints["footprint_id"].is_unique
assert footprints["source"].value_counts().to_dict() == {"DAT": 16465, "SED": 8393}
assert footprints.crs.to_epsg() == 4326 or footprints.crs.to_string() == "OGC:CRS84"
assert footprints.geometry.notna().all() and footprints.geometry.is_valid.all()
coverage = footprints.groupby(["source_year", "source"]).size().unstack(fill_value=0)
display(coverage)
coverage.plot.bar(stacked=True, figsize=(12, 4), color=["#277e8e", "#d4a445"])
plt.title("EFC footprint regions by source and annual partition")
plt.xlabel("Source year")
plt.ylabel("Regions, not unique tornadoes")
plt.tight_layout()
plt.show()
quality = pd.Series({
    "unknown_or_nonstandard_EF": footprints["ef_rating"].isna().sum(),
    "placeholder_width_0.99": footprints["width_is_placeholder"].sum(),
    "unusable_width": footprints["path_width_yards"].isna().sum(),
    "regions_with_parents": footprints["parents"].map(len).gt(0).sum(),
    "regions_with_children": footprints["children"].map(len).gt(0).sum(),
}, name="region_count")
display(quality.to_frame())
display(footprints[["footprint_id", "source_year", "source", "efscale", "ef_rating",
                    "max_ef_rating", "width", "path_width_yards", "parents", "children"]].head())

### A local footprint example

The map below selects May 20, 2013 footprint regions intersecting a window around Moore, Oklahoma. It illustrates source geometry and possible overlap, not an independently measured wind field or an automatically matched SPC track. Reconstructed footprints elsewhere can have different accuracy.

In [ ]:
example = footprints.loc[
    footprints["stormdate_datetime_utc"].dt.strftime("%Y-%m-%d").eq("2013-05-20")
].cx[-97.65:-97.35, 35.2:35.5]
assert not example.empty
fig, ax = plt.subplots(figsize=(10, 6))
example.to_crs("EPSG:5070").plot(
    ax=ax, column="ef_rating", categorical=True, legend=True,
    alpha=0.6, edgecolor="#333333", linewidth=0.4,
    missing_kwds={"color": "lightgrey", "label": "Unknown"},
)
ax.set_title("Example EFC damage regions: Moore area, May 20, 2013")
ax.set_axis_off()
plt.tight_layout()
plt.show()
display(example[["footprint_id", "source", "ef_rating", "max_ef_rating"]])

## Census: county context and map

Population and housing units are county totals, not counts struck or individual building locations. `county_fips` stays a five-character string. Estimates use vintage 2020 for 2010–2019 and vintage 2025 for 2020–2025.

The fixed 2020 map is generalized to 1:5,000,000 and is unsuitable for precise damage-path exposure calculations. New Connecticut planning-region codes are absent from it; matching other FIPS codes does not prove boundaries remained unchanged.

In [ ]:
assert len(counties) == 50294
assert counties["county_fips"].str.fullmatch(r"\d{5}").all()
display(counties.head())
display(counties.loc[~counties["map_2020_fips_present"],
                     ["county_fips", "state_name", "county_name"]].drop_duplicates())
fig, ax = plt.subplots(figsize=(13, 7))
boundaries.cx[-126:-66, 24:50].boundary.plot(ax=ax, color="#c5ccd1", linewidth=0.3)
conus = spc.loc[spc["start_longitude"].between(-126, -66) & spc["start_latitude"].between(24, 50)]
ax.scatter(conus["start_longitude"], conus["start_latitude"], s=2, alpha=0.35,
           color="#277e8e", linewidths=0)
ax.set(xlim=(-126, -66), ylim=(24, 50), xlabel="Longitude", ylabel="Latitude",
       title="SPC tornado start locations, 2010–2025: contiguous U.S. view")
ax.set_aspect(1.25)
plt.tight_layout()
plt.show()

## Before modeling EF ratings

- Start with SPC's tornado rows and make unknown-rating exclusions explicit. Only eight tracks have EF5 labels; class imbalance is severe.
- Define when predictions are meant to be made. Damage, casualties, narratives, survey wind estimates, and footprint rating/shape information can leak EF labels or encode post-event information.
- Match sources explicitly and keep records from the same tornado or related outbreak together when splitting data. The table IDs alone do not join SPC, NCEI and EFC.
- County context is broad. Footprints do not give independently verified exposure counts, and better documented damage can affect observed ratings.
- This dataset does not contain radar or environmental inputs for forecasting intensity before damage occurs.

The seven tables remain separate to preserve their units and provenance. Full original files are available separately in `release.zip.bin`, and as direct source files on the [Hugging Face mirror](https://huggingface.co/datasets/jakeryderv/us-tornado-data-2010-2025/tree/v2.0.0).

Credit NOAA/NWS SPC, NOAA NCEI, NOAA/NWS DAT through the Footprint Catalog, and the U.S. Census Bureau. See the [source and reuse notes](https://github.com/jakeryderv/us-tornado-data-2010-2025/blob/a84ca6f4f8382788b7952e4b63d151696684ea59/docs/DATA_SOURCES.md).

In [ ]:
assert annual["spc_tracks"].sum() == len(spc)
assert annual["efc_dat_footprints"].sum() == footprints["source"].eq("DAT").sum()
assert annual["efc_sed_footprints"].sum() == footprints["source"].eq("SED").sum()
assert len(tables) == 8 and len(verified_paths) == 9
print("Inspection passed: v2.0.0 / Kaggle 5; seven main tables plus annual summary.")
print("Release manifest and all nine selected analysis files passed SHA-256 checks.")
print("Full-collection integrity and cross-source tornado matching are outside this notebook's scope.")